### **Advanced Retrieval Techniques**

These are advanced methods to improve retrieval quality beyond basic similarity search.

They focus on combining multiple retrievers, transforming queries, and extracting better context.

### **1) Fusion Methods**

#### **1A) Weighted Fusion**

**Concept:** Combine results from multiple retrievers by assigning weights to each.

Instead of using only one retriever, you run multiple retrievers and merge their results.

Each retriever gets a weight - more important retrievers get higher weights.

**Example:**
- BM25 retriever (sparse) with weight 0.4
- Dense retriever (embedding-based) with weight 0.6

Final score = 0.4 × BM25_score + 0.6 × Dense_score

**When to use:** When you have multiple good retrieval methods and want to leverage their strengths together.

#### **1B) Reciprocal Rank Fusion (RRF)**

**Concept:** Combine rankings from multiple retrievers using reciprocal ranks.

Instead of comparing scores (which may be on different scales), use the **rank** of each document.

**Formula:** Score = 1/(k + rank)

where k is a constant (usually 60) to avoid division issues.

**Why it's useful:**
- Different retrievers may use different scoring scales
- RRF normalizes everything to ranks
- Simple and doesn't require tuning weights

**Example:**
- Document A: rank 1 in BM25, rank 5 in Dense → RRF score = 1/61 + 1/65 ≈ 0.0313
- Document B: rank 3 in BM25, rank 1 in Dense → RRF score = 1/63 + 1/61 ≈ 0.0328

**When to use:** When combining multiple retrievers and you want a simple, parameter-free approach.

### **2) Query Transformation Retrievers**

#### **2A) Multi Query Retriever**

**Concept:** Instead of using the original query, generate multiple variations of the same query and retrieve documents for each.

**How it works:**
1. User asks a question
2. LLM generates 3-5 variations of the same question
3. Run retrieval for each variation
4. Combine results (remove duplicates)

**Example:**
- Original: "How do neural networks work?"
- Variation 1: "Explain the architecture of neural networks"
- Variation 2: "What are the layers in a neural network?"
- Variation 3: "How do forward passes work in neural networks?"

**Benefit:** Catches documents that match different phrasings of the same intent.

**When to use:** When a single query might not capture all relevant documents due to different terminology.

#### **2B) HYDE (Hypothetical Document Embeddings)**

**Concept:** Generate a hypothetical answer to the query, then use that answer to retrieve documents.

**Why it's different:** Instead of searching for documents similar to the query, search for documents similar to a hypothetical response.

**How it works:**
1. User asks: "What causes climate change?"
2. LLM generates a hypothetical answer: "Climate change is primarily caused by greenhouse gas emissions from human activities. CO2 from burning fossil fuels traps heat..."
3. Embed the hypothetical answer
4. Retrieve documents similar to this answer

**Why this helps:** The hypothetical answer is often longer and richer than the query, providing better semantic context.

**When to use:** When you have short queries that lack context, or when the answer is very different from the question phrasing.

#### **2C) Multi Hop Retriever**

**Concept:** Retrieve documents iteratively - use information from previous retrievals to refine subsequent searches.

**How it works:**
1. First retrieval: Get initial documents for the query
2. Extract entities/concepts from those documents
3. Second retrieval: Search for more documents using the extracted concepts
4. Repeat as needed

**Example:**
- Query: "Who invented Python and why?"
- Hop 1: Find documents about Python programming language
- Extract: "Guido van Rossum" mentioned
- Hop 2: Find documents about Guido van Rossum
- Hop 3: Find documents about his motivation and design decisions

**When to use:** When questions require reasoning across multiple pieces of information, or when you need to follow chains of related documents.

### **3) Context Optimization Retrievers**

#### **3A) Parent Document Retriever**

**Problem:** Chunks are often too small to be meaningful when retrieved individually.

**Concept:** Index small chunks for retrieval, but return their parent (larger) documents.

**How it works:**
1. Create small chunks (for similarity search precision)
2. Link each chunk to its parent document
3. Search using small chunks
4. Return the full parent document instead of the chunk

**Example:**
- Book: "The Origin of Species" (parent)
- Chapter 3: "Natural Selection" (medium chunk)
- Sentence about finches (small chunk for indexing)
- Search matches the sentence, return the whole chapter

**Benefit:** Small chunks find the right content, but you get context from the full parent.

**When to use:** When your documents are hierarchical (books → chapters → sections) or when small chunks lose important context.

#### **3B) Sentence Window Retriever**

**Problem:** A retrieved sentence might not have enough context to be useful.

**Concept:** Index individual sentences, but return a window of sentences around the match.

**How it works:**
1. Break text into sentences
2. Embed each sentence separately for precise matching
3. When a sentence is retrieved, return it along with N surrounding sentences

**Example:**
- Sentences in document: [S1, S2, S3, S4, S5]
- Query matches S3
- Return: [S2, S3, S4] as context (window of 1 on each side)

**Benefit:** Fine-grained retrieval with better context.

**When to use:** When you want precise retrieval but single sentences are too minimal, similar to Parent Document but more granular.

#### **3C) Contextual Compression**

**Problem:** Retrieved documents may contain irrelevant information alongside relevant content.

**Concept:** After retrieval, use an LLM to compress and extract only relevant parts from the retrieved documents.

**How it works:**
1. Retrieve documents normally
2. Pass the query + retrieved documents to an LLM
3. LLM extracts/summarizes only the relevant parts
4. Return the compressed version

**Example:**
- Retrieved: "Chapter on Climate Change (2000 words)"
- Query: "What year did the Kyoto Protocol start?"
- Compressed: "The Kyoto Protocol was adopted in 1997 and came into effect in 2005..."

**Benefit:** Reduces irrelevant information, makes answers more precise.

**Trade-off:** Requires an additional LLM call, adding latency.

**When to use:** When context length is limited or when retrieved documents are noisy with irrelevant information.

### **4) Quick Comparison**

| Technique | Problem Solved | Complexity | When to Use |
|-----------|---|---|---|
| Weighted Fusion | Single retriever has limitations | Low | Multiple retrievers available |
| RRF | Different scoring scales | Low | No weights to tune |
| Multi Query | Query variations miss docs | Medium | Short, ambiguous queries |
| HYDE | Questions are too short | Medium | Limited query context |
| Multi Hop | Needs multi-step reasoning | High | Complex, multi-entity questions |
| Parent Document | Chunks lack context | Low-Medium | Hierarchical documents |
| Sentence Window | Single sentences too minimal | Low-Medium | Fine-grained retrieval needed |
| Contextual Compression | Retrieved docs have noise | Medium | Limited context window |

### **5) Key Takeaway**

The goal of these techniques is simple: **Get the right information to the LLM in the right form.**

Some address the retrieval phase (fusion, multi-hop, multi-query), others optimize the context (window, parent, compression).

In practice, you often combine multiple techniques for best results.